# FMSE Lab 01 - Build a Typed Model Call

**Course:** FMSE 2026 · **Module 1:** Engineering Bootcamp for AI Systems · **Phase:** Foundation · **Tier:** Foundation

Course home and progress: [agentic-ai.es/academy/fmse/learn/m01-engineering-bootcamp](https://agentic-ai.es/academy/fmse/learn/m01-engineering-bootcamp).

## 0. Mission and competency target

**Mission.** Put a typed, tested boundary around a model call: a timeout, a bounded retry, schema validation and a deterministic business rule — so every failure comes back as a typed result your code can act on, and the credential never leaks.

**Competency target (Module 1 outcomes):**
- Use Python notebooks, environment variables/secrets, HTTP APIs, JSON, JSON Schema, and typed validation.
- Explain deterministic versus probabilistic components and where each belongs in a production AI system.
- Write small automated tests and use Git/GitHub concepts needed for later labs.
- Measure latency, token usage, cost, retries, and failure rate for a model call.

**Scaffolding:** Foundation — about 70% guided, 30% independent. Starter signatures are provided; the implementation is not.

## 1. Requirements and acceptance criteria

| ID | Requirement |
| --- | --- |
| API-01 | Valid response passes |
| API-02 | Malformed JSON fails gracefully |
| API-03 | Schema mismatch returns typed error |
| API-04 | Retry count is bounded |
| API-05 | Business-rule violation returns typed error |
| SEC-01 | Secret is not printed or persisted in notebook output **(critical)** |
| MAN-01 | Experiment manifest exported |

**Competency gate (portal):** Quiz >= 80%; all public tests PASS; experiment manifest exported. A critical requirement cannot be offset by other scores.

## 2. Environment setup

In [ ]:
# Idempotent: safe to re-run, and it never prints secrets.
LAB_ID = "lab-01"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit already loaded"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"using the labkit in {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "using the previously downloaded labkit"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"downloaded and verified the labkit from {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "Could not load fmse_labkit from: " + "; ".join(problems) + ". "
        "Recovery: download the labs folder from agentic-ai.es/academy/fmse/resources, upload the fmse_labkit folder next to this notebook (Files panel), and re-run this cell."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("en")
print(f"fmse_labkit {fmse.__version__} ready for {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Secrets check

This lab uses a **mock** model endpoint and a fake credential generated by the validator, so no real key is needed. Real keys, when you use them in later labs, live in Colab Secrets — never in a cell, in markdown, in Git history, or in a URL.

In [ ]:
fmse.secrets_check()

## 4. Guided experiment

`MockModelEndpoint` behaves like a real HTTP model service: it needs an `Authorization` header and a `timeout_s`, and it can return malformed JSON, schema-violating JSON, overload errors (HTTP 503), or time out. `simple_call` is a working wrapper for the happy path: one call, parse, validate against the schema.

In [ ]:
import json
from fmse_labkit.labs import lab01

guided = fmse.GuidedLog(LAB_ID, required_steps=["happy_path", "inspect_failures", "schema_inspection"])
DEMO_KEY = "demo-key-not-a-secret"

print("Contract (JSON Schema):")
print(json.dumps(lab01.INVOICE_SCHEMA, indent=2))
print("Business rules:", lab01.BUSINESS_RULES)

data = lab01.simple_call(lab01.MockModelEndpoint("valid"), lab01.REQUEST, DEMO_KEY)
print("\nHappy path result:", data)
guided.step("happy_path")

## 5. Predict before you run

The next cell runs `simple_call` against every failure scenario. For **malformed_json** and **always_503**: what will happen — a typed result, an exception, or a hang? How many calls will the 503 case make?

In [ ]:
my_prediction = ""
guided.predict("simple_call_failures", my_prediction)

In [ ]:
observed = {}
for scenario in ["malformed_json", "schema_mismatch", "business_rule", "always_503", "timeout"]:
    endpoint = lab01.MockModelEndpoint(scenario)
    try:
        lab01.simple_call(endpoint, lab01.REQUEST, DEMO_KEY)
        observed[scenario] = f"returned normally (!) after {endpoint.calls} call(s)"
    except Exception as exc:
        observed[scenario] = f"raised {type(exc).__name__} after {endpoint.calls} call(s)"
    print(f"{scenario:<16} {observed[scenario]}")
guided.step("inspect_failures")
guided.outcome("simple_call_failures", observed)

Note **business_rule**: the response is schema-valid, and `simple_call` accepts it — yet its total does not equal its line items. Syntactic validity, schema validity and semantic correctness are three different levels (lesson 1.3).

In [ ]:
bad = json.loads(lab01.MockModelEndpoint("business_rule").post(lab01.REQUEST, headers={"Authorization": "Bearer x"}, timeout_s=1).text)
print("schema errors:", fmse.schema.validate(bad, lab01.INVOICE_SCHEMA))
print("total:", bad["total"], " sum of lines:", sum(i["amount"] for i in bad["line_items"]))
guided.step("schema_inspection")

In [ ]:
worksheet = {
    "failure_modes": "",  # Which failures raised, and which were silently accepted?
    "retry_question": "",  # Which of these failures are worth retrying, and which are not? Why?
    "prediction_check": "",  # Was your prediction right?
}
guided.observe(worksheet)

## 6. Challenge

Implement `call_and_validate()` with **timeout, bounded retry, schema validation, and deterministic business-rule checks**. It must return a `lab01.CallResult` whose `status` is one of:

`ok` · `invalid_json` · `schema_error` · `business_rule_error` · `transport_error` · `timeout`

and whose `attempts` counts the calls you made. Then build an **experiment manifest** (lesson 1.4) using the metrics from `lab01.run_experiment(call_and_validate)`.

The grader calls your function with a credential it generates itself and watches everything your code prints, returns and raises.

In [ ]:
from fmse_labkit.labs.lab01 import CallResult


def call_and_validate(endpoint, request, schema, *, api_key, timeout_s=5.0, max_retries=2) -> CallResult:
    """Call endpoint.post(request, headers=..., timeout_s=...) and return a typed CallResult."""
    raise NotImplementedError

In [ ]:
experiment_metrics = lab01.run_experiment(call_and_validate)
print(experiment_metrics)

manifest = {
    "model": "",
    "prompt_version": "",
    "dataset_version": "",
    "metrics": {},
}

## 7. Public validation

In [ ]:
submission = {"call_and_validate": call_and_validate, "manifest": manifest}
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustness / adversarial probes

Conditions beyond the public suite: different retry bounds, and an error that retrying cannot fix.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Engineering reflection

Graded in the portal. Draft here if useful:
- **Decision:** which failures did you retry and which did you not, and why?
- **Trade-offs:** what did your retry bound and timeout trade against each other (cost, latency, success rate)?
- **Remaining risks:** what can still be wrong in a response that passes your schema and business rules?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Export artifact / completion result

The artifact is your **Typed AI Interface Contract + tested Python wrapper**: the schema, the business rules, your function's source, and the experiment manifest.

In [ ]:
import inspect

try:
    wrapper_source = inspect.getsource(call_and_validate)
except (OSError, TypeError):
    wrapper_source = "# source unavailable in this runtime"

contract_artifact = {
    "contract": {"schema": lab01.INVOICE_SCHEMA, "business_rules": lab01.BUSINESS_RULES, "statuses": list(lab01.STATUSES)},
    "wrapper_source": wrapper_source,
    "experiment_manifest": manifest,
    "public_checks": {c.id: c.status for c in result.checks},
}
artifact = fmse.export_artifact(LAB_ID, contract_artifact, title="Typed AI Interface Contract + tested Python wrapper", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, "experiment_metrics": experiment_metrics},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)